# 03 - Submission
Membuat `submission.csv` dari model tersimpan (hasil `02_modeling`), memvalidasi formatnya, lalu mengirimkannya ke Kaggle.

Format yang diminta: kolom `building_id,damage_grade` untuk setiap baris `test.csv`.

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

sys.path.insert(0, "..")      # agar folder scripts/ bisa di-import dari notebooks/

import pandas as pd

from scripts.predict import COMPETITION, make_submission, submit_to_kaggle

# ---- lokasi data dan output (otomatis: Kaggle atau lokal) ----
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob("**/train.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:                                   # lokal: csv di ../data, output ke root repo
    DATA_DIR = Path("../data")
    OUT_DIR = Path("..")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

# model yang dipakai = model terakhir yang disimpan oleh 02_modeling
MODELS_DIR = OUT_DIR / "models"
latest = max(MODELS_DIR.glob("pipeline_model_*.pkl"), key=lambda p: p.stat().st_mtime)
MODEL_NAME = latest.stem.replace("pipeline_model_", "")
MODEL_PATH = latest
WEIGHTS_PATH = MODELS_DIR / f"class_weights_{MODEL_NAME}.json"
OUTPUT = OUT_DIR / "submission.csv" if ON_KAGGLE else OUT_DIR / "submissions" / "submission.csv"
print("Model:", MODEL_NAME)

## 1. Prediksi test dan simpan `submission.csv`

In [ ]:
sub = make_submission(MODEL_PATH, DATA_DIR, OUTPUT, WEIGHTS_PATH)
sub.head()

## 2. Cek format

In [ ]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Kolom sama     :", list(sub.columns) == list(sample.columns))
print("Jumlah baris   :", len(sub), "vs sample", len(sample))
print("ID identik     :", set(sub["building_id"]) == set(sample["building_id"]))
print("Nilai grade    :", sorted(sub["damage_grade"].unique()))
sub["damage_grade"].value_counts(normalize=True).sort_index().round(3)